# Sweden Energy Pipeline — Phase 4 EDA

Scratch-space exploration before writing `analyse.py`. Cells are meant to be run top-to-bottom after the pipeline has ingested at least a few days of data.

**DB connection** reads from `.env` in the repo root (same as the pipeline).

In [ ]:
import sys
from pathlib import Path

# Add repo root to path so pipeline.db is importable
repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.stats import pearsonr

from pipeline.db import get_conn

conn = get_conn()
print('Connected to DB')

## 1. Load `hourly_energy` view

In [ ]:
hourly = pd.read_sql('SELECT * FROM hourly_energy ORDER BY hour', conn)
hourly['hour'] = pd.to_datetime(hourly['hour'], utc=True)
print(f'{len(hourly)} rows, {hourly["hour"].min()} → {hourly["hour"].max()}')
hourly.head()

In [ ]:
# Quick sanity check on numeric columns
hourly[['price_eur_mwh', 'temperature_c', 'windspeed_ms', 'radiation_wm2',
        'greenness_score', 'total_gen_mw']].describe()

## 2. Price over time — all zones

In [ ]:
prices_all = pd.read_sql(
    'SELECT price_time AS hour, zone, price_eur_mwh FROM raw_prices ORDER BY price_time',
    conn
)
prices_all['hour'] = pd.to_datetime(prices_all['hour'], utc=True)

fig, ax = plt.subplots(figsize=(14, 4))
for zone, grp in prices_all.groupby('zone'):
    ax.plot(grp['hour'], grp['price_eur_mwh'], label=zone, linewidth=0.8)
ax.set_title('Spot price (EUR/MWh) by bidding zone')
ax.set_ylabel('EUR/MWh')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 3. Windspeed vs price — scatter coloured by zone

In [ ]:
# Join prices with Stockholm weather for all zones
weather_sql = """
    SELECT forecast_time AS hour, temperature_c, windspeed_ms, radiation_wm2
    FROM raw_weather
    WHERE city = 'stockholm' AND source = 'openmeteo'
"""
weather = pd.read_sql(weather_sql, conn)
weather['hour'] = pd.to_datetime(weather['hour'], utc=True)

scatter_df = prices_all.merge(weather, on='hour', how='inner')

zones = scatter_df['zone'].unique()
colours = plt.cm.tab10.colors

fig, ax = plt.subplots(figsize=(8, 5))
for i, zone in enumerate(sorted(zones)):
    sub = scatter_df[scatter_df['zone'] == zone]
    ax.scatter(sub['windspeed_ms'], sub['price_eur_mwh'],
               label=zone, alpha=0.5, s=15, color=colours[i])
ax.set_xlabel('Wind speed (m/s) — Stockholm')
ax.set_ylabel('Price (EUR/MWh)')
ax.set_title('Wind speed vs spot price by zone')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Pearson r — windspeed vs price

In [ ]:
print('Pearson r: windspeed_ms vs price_eur_mwh')
print('-' * 42)
for zone in sorted(scatter_df['zone'].unique()):
    sub = scatter_df[scatter_df['zone'] == zone][['windspeed_ms', 'price_eur_mwh']].dropna()
    if len(sub) < 3:
        print(f'  {zone}: insufficient data ({len(sub)} rows)')
        continue
    r, p = pearsonr(sub['windspeed_ms'], sub['price_eur_mwh'])
    print(f'  {zone}: r = {r:+.3f}  (n={len(sub)}, p={p:.3f})')

## 5. Greenness score over 24 hours — SE3

In [ ]:
se3 = hourly.dropna(subset=['greenness_score']).copy()

fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(se3['hour'], se3['greenness_score'], color='green', linewidth=0.9, alpha=0.8)
ax.axhline(se3['greenness_score'].mean(), color='darkgreen', linestyle='--',
           linewidth=0.8, label=f'Mean = {se3["greenness_score"].mean():.1f}%')
ax.set_title('Greenness score (%) — SE3 actual generation')
ax.set_ylabel('%')
ax.set_ylim(0, 105)
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print(f'Greenness range: {se3["greenness_score"].min():.1f}% – {se3["greenness_score"].max():.1f}%')

## 6. Average price by hour-of-day — SE3

In [ ]:
se3_all = prices_all[prices_all['zone'] == 'SE3'].copy()
se3_all['hour_of_day'] = se3_all['hour'].dt.hour

avg_by_hour = se3_all.groupby('hour_of_day')['price_eur_mwh'].mean()

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(avg_by_hour.index, avg_by_hour.values,
              color=plt.cm.RdYlGn_r(avg_by_hour / avg_by_hour.max()))
ax.set_xticks(range(24))
ax.set_xlabel('Hour of day (UTC)')
ax.set_ylabel('Avg price (EUR/MWh)')
ax.set_title('Average spot price by hour of day — SE3')
plt.tight_layout()
plt.show()

print('Cheapest 3 hours:', avg_by_hour.nsmallest(3).to_dict())
print('Costliest 3 hours:', avg_by_hour.nlargest(3).to_dict())

## 7. Top 3 and bottom 3 hours by appliance signal (from features_hourly)

In [ ]:
try:
    bh = pd.read_sql(
        "SELECT hour_of_day, combined_score FROM features_best_hours "
        "WHERE zone = 'SE3' ORDER BY combined_score DESC",
        conn
    )
    if bh.empty:
        print('features_best_hours is empty — run analyse.py first')
    else:
        print('Top 3 hours to run appliances (SE3):')
        print(bh.head(3).to_string(index=False))
        print()
        print('Bottom 3 hours (avoid):')
        print(bh.tail(3).to_string(index=False))
except Exception as exc:
    print(f'features_best_hours not yet populated: {exc}')

In [ ]:
# Appliance signal distribution
try:
    sig_df = pd.read_sql(
        "SELECT appliance_signal, COUNT(*) AS n FROM features_hourly GROUP BY appliance_signal",
        conn
    )
    print('Appliance signal distribution:')
    print(sig_df.to_string(index=False))
except Exception as exc:
    print(f'features_hourly not yet populated: {exc}')

conn.close()